# Bulk Data Coverage Analysis

This notebook analyzes dataset coverage by reading parquet files directly from S3.

It provides descriptive statistics about data coverage including:
- Row counts by partition
- Geographic coverage (states, zip codes)
- Temporal coverage (date ranges)
- Category distributions

## Setup

In [ ]:
import boto3
import pandas as pd
import time
from io import BytesIO
from typing import Optional, Dict, List

In [ ]:
# Configuration
BUCKET = "carc-ext-ant"
REGION = "us-east-1"

# Initialize AWS clients
s3 = boto3.client("s3", region_name=REGION)
athena = boto3.client("athena", region_name=REGION)

print(f"Connected to bucket: {BUCKET}")

## 1. List Available Datasets

Discover what datasets are available in the bucket.

In [ ]:
def list_datasets(bucket: str) -> List[str]:
    """List available datasets (top-level prefixes) in the bucket."""
    response = s3.list_objects_v2(
        Bucket=bucket,
        Delimiter="/",
        MaxKeys=100
    )
    
    prefixes = []
    for prefix in response.get("CommonPrefixes", []):
        prefixes.append(prefix["Prefix"].rstrip("/"))
    return prefixes

datasets = list_datasets(BUCKET)
print("Available data drops:")
for ds in datasets:
    print(f"  - {ds}")

## 2. Explore Dataset Structure

Drill down into a specific dataset to understand its structure.

In [ ]:
def list_prefix_contents(bucket: str, prefix: str, max_items: int = 20):
    """List contents of a prefix."""
    response = s3.list_objects_v2(
        Bucket=bucket,
        Prefix=prefix,
        Delimiter="/",
        MaxKeys=max_items
    )
    
    print(f"Contents of {prefix}:")
    print("=" * 60)
    
    for p in response.get("CommonPrefixes", []):
        print(f"  [DIR]  {p['Prefix']}")
    
    for obj in response.get("Contents", []):
        size_mb = obj["Size"] / (1024 * 1024)
        print(f"  [FILE] {obj['Key']} ({size_mb:.2f} MB)")

In [ ]:
# Explore the latest date folder
list_prefix_contents(BUCKET, "20260108/")

In [ ]:
# Explore deeper
list_prefix_contents(BUCKET, "20260108/Full/")

## 3. Discover Schema

Read a sample parquet file to understand the data structure.

In [ ]:
def discover_schema(bucket: str, prefix: str) -> Dict:
    """
    Discover the schema by reading a sample parquet file.
    
    Args:
        bucket: S3 bucket name
        prefix: S3 prefix to search for parquet files
        
    Returns:
        Dictionary with schema information
    """
    # Find a parquet file
    paginator = s3.get_paginator("list_objects_v2")
    parquet_key = None
    
    for page in paginator.paginate(Bucket=bucket, Prefix=prefix):
        for obj in page.get("Contents", []):
            if obj["Key"].endswith(".parquet") and obj["Size"] > 1000:
                parquet_key = obj["Key"]
                break
        if parquet_key:
            break
    
    if not parquet_key:
        raise ValueError(f"No parquet files found under prefix: {prefix}")
    
    # Read the parquet file
    response = s3.get_object(Bucket=bucket, Key=parquet_key)
    df = pd.read_parquet(BytesIO(response["Body"].read()))
    
    schema = {
        "sample_file": parquet_key,
        "row_count": len(df),
        "columns": {}
    }
    
    for col in df.columns:
        schema["columns"][col] = {
            "dtype": str(df[col].dtype),
            "null_count": int(df[col].isnull().sum()),
            "unique_count": int(df[col].nunique()),
            "sample_values": df[col].dropna().head(3).tolist()
        }
    
    return schema, df

In [ ]:
# Set the prefix to analyze
DATA_PREFIX = "20260108/Full/ant_permit_data"

schema, sample_df = discover_schema(BUCKET, DATA_PREFIX)

print(f"Sample file: {schema['sample_file']}")
print(f"Rows: {schema['row_count']:,}")
print("\nColumns:")
print("=" * 60)
for col, info in schema["columns"].items():
    print(f"  {col:30} {info['dtype']:10} ({info['unique_count']:,} unique)")

In [ ]:
# Preview sample data
sample_df.head(10)

## 4. Coverage Statistics

Sample multiple parquet files to get comprehensive coverage statistics.

In [ ]:
def get_coverage_stats(bucket: str, prefix: str, sample_size: int = 5) -> Dict:
    """
    Get coverage statistics by sampling multiple parquet files.
    
    Args:
        bucket: S3 bucket name
        prefix: S3 prefix to analyze
        sample_size: Number of parquet files to sample
        
    Returns:
        Dictionary with coverage statistics
    """
    # Find parquet files
    paginator = s3.get_paginator("list_objects_v2")
    parquet_files = []
    
    for page in paginator.paginate(Bucket=bucket, Prefix=prefix):
        for obj in page.get("Contents", []):
            if obj["Key"].endswith(".parquet") and obj["Size"] > 1000:
                parquet_files.append({
                    "key": obj["Key"],
                    "size": obj["Size"]
                })
    
    if not parquet_files:
        raise ValueError(f"No parquet files found under prefix: {prefix}")
    
    print(f"Found {len(parquet_files):,} parquet files")
    
    # Sample files (take largest ones for better coverage)
    parquet_files.sort(key=lambda x: x["size"], reverse=True)
    sample_files = parquet_files[:sample_size]
    
    # Read and combine samples
    dfs = []
    for i, f in enumerate(sample_files):
        try:
            print(f"  Reading file {i+1}/{sample_size}: {f['key'].split('/')[-1]}")
            response = s3.get_object(Bucket=bucket, Key=f["key"])
            df = pd.read_parquet(BytesIO(response["Body"].read()))
            dfs.append(df)
        except Exception as e:
            print(f"  Warning: Could not read {f['key']}: {e}")
    
    if not dfs:
        raise ValueError("Could not read any parquet files")
    
    combined = pd.concat(dfs, ignore_index=True)
    
    stats = {
        "files_sampled": len(dfs),
        "total_files": len(parquet_files),
        "sampled_rows": len(combined),
        "columns": list(combined.columns),
        "dataframe": combined,
    }
    
    # Add column-specific stats
    stats["column_stats"] = {}
    for col in combined.columns:
        col_stats = {
            "dtype": str(combined[col].dtype),
            "null_pct": round(combined[col].isnull().mean() * 100, 2),
            "unique_count": int(combined[col].nunique()),
        }
        
        # Add value distribution for categorical columns
        if combined[col].dtype == "object" or combined[col].nunique() < 50:
            col_stats["value_counts"] = combined[col].value_counts().head(10).to_dict()
        
        # Add numeric stats for numeric columns
        if pd.api.types.is_numeric_dtype(combined[col]):
            col_stats["min"] = float(combined[col].min()) if not pd.isna(combined[col].min()) else None
            col_stats["max"] = float(combined[col].max()) if not pd.isna(combined[col].max()) else None
            col_stats["mean"] = float(combined[col].mean()) if not pd.isna(combined[col].mean()) else None
        
        stats["column_stats"][col] = col_stats
    
    return stats

In [ ]:
# Get coverage stats (sample 3 large files)
stats = get_coverage_stats(BUCKET, DATA_PREFIX, sample_size=3)

print(f"\nSampled {stats['files_sampled']} / {stats['total_files']} files")
print(f"Total rows: {stats['sampled_rows']:,}")

## 5. Coverage Report

Generate detailed coverage statistics.

In [ ]:
def print_coverage_report(stats: Dict):
    """Print a formatted coverage report."""
    print("\n" + "=" * 70)
    print("BULK DATA COVERAGE REPORT")
    print("=" * 70)
    
    print(f"\nFiles sampled: {stats['files_sampled']} / {stats['total_files']}")
    print(f"Rows sampled: {stats['sampled_rows']:,}")
    print(f"Columns: {len(stats['columns'])}")
    
    print("\n" + "-" * 70)
    print("COLUMN STATISTICS")
    print("-" * 70)
    
    for col, col_stats in stats["column_stats"].items():
        print(f"\n{col}:")
        print(f"  Type: {col_stats['dtype']}")
        print(f"  Null %: {col_stats['null_pct']}%")
        print(f"  Unique values: {col_stats['unique_count']:,}")
        
        if "min" in col_stats:
            print(f"  Range: {col_stats['min']} to {col_stats['max']}")
            if col_stats["mean"]:
                print(f"  Mean: {col_stats['mean']:.2f}")
        
        if "value_counts" in col_stats:
            print("  Top values:")
            for val, count in list(col_stats["value_counts"].items())[:5]:
                print(f"    {val}: {count:,}")

print_coverage_report(stats)

## 6. Geographic Coverage

In [ ]:
df = stats["dataframe"]

if "state" in df.columns:
    print("Geographic Coverage by State:")
    print("=" * 40)
    state_counts = df["state"].value_counts()
    print(f"Total states: {len(state_counts)}")
    print(f"\nTop 10 states:")
    print(state_counts.head(10).to_string())

In [ ]:
if "zip_code" in df.columns:
    print("\nZip Code Coverage:")
    print("=" * 40)
    print(f"Total unique zip codes: {df['zip_code'].nunique():,}")
    print(f"\nTop 10 zip codes:")
    print(df["zip_code"].value_counts().head(10).to_string())

## 7. Category Distribution

In [ ]:
if "permit_classifier_name" in df.columns:
    print("Permit Type Distribution:")
    print("=" * 60)
    permit_counts = df["permit_classifier_name"].value_counts()
    print(permit_counts.to_string())

In [ ]:
if "building_permit_status_name" in df.columns:
    print("\nPermit Status Distribution:")
    print("=" * 40)
    status_counts = df["building_permit_status_name"].value_counts()
    print(status_counts.to_string())

## 8. Numeric Field Analysis

In [ ]:
if "permit_job_value" in df.columns:
    print("Permit Job Value Statistics:")
    print("=" * 40)
    print(df["permit_job_value"].describe().to_string())
    
    # Distribution by range
    print("\nJob Value Distribution:")
    bins = [0, 1000, 5000, 10000, 50000, 100000, 500000, float('inf')]
    labels = ['$0-1K', '$1K-5K', '$5K-10K', '$10K-50K', '$50K-100K', '$100K-500K', '$500K+']
    df['value_range'] = pd.cut(df['permit_job_value'], bins=bins, labels=labels)
    print(df['value_range'].value_counts().sort_index().to_string())

In [ ]:
if "permit_fees" in df.columns:
    print("\nPermit Fees Statistics:")
    print("=" * 40)
    print(df["permit_fees"].describe().to_string())

## 9. Summary

In [ ]:
print("=" * 70)
print("COVERAGE SUMMARY")
print("=" * 70)
print(f"\nBucket: {BUCKET}")
print(f"Prefix: {DATA_PREFIX}")
print(f"\nTotal parquet files: {stats['total_files']:,}")
print(f"Files sampled: {stats['files_sampled']}")
print(f"Rows sampled: {stats['sampled_rows']:,}")

if "state" in df.columns:
    print(f"\nStates covered: {df['state'].nunique()}")
if "zip_code" in df.columns:
    print(f"Zip codes covered: {df['zip_code'].nunique():,}")
if "permit_classifier_name" in df.columns:
    print(f"Permit types: {df['permit_classifier_name'].nunique()}")
if "building_permit_status_name" in df.columns:
    print(f"Permit statuses: {df['building_permit_status_name'].nunique()}")